# AMD GPU Local Stable Diffusion & Document Editing Pipeline
This notebook provides a complete environment and code for running the document forgery detection and layout editing pipeline locally on Windows with an **AMD GPU** using the **DirectML** hardware acceleration backend.

### Key Capabilities:
1. **AMD GPU Acceleration**: Uses Microsoft's `torch-directml` library to leverage DirectX 12 hardware acceleration on AMD GPUs.
2. **Self-Hosted Image Generation**: Replaces remote OpenRouter API calls with a local **Stable Diffusion Image-to-Image (Img2Img)** pipeline.
3. **Image Authenticity & Noise Replication**: Feeds the original cropped layout block (with its noise, compression artifacts, and texture) back into the Stable Diffusion pipeline along with a text prompt to generate a style-matched replacement.
4. **Inline Cell Visualizations**: Draws block bounding boxes and block IDs on the image and displays them inline so you know exactly which areas can be replaced without needing an external UI.

## Step 1: Install Dependencies
Run the following cell to install the required libraries. 

> **Note:** `torch-directml` is the core package that binds PyTorch to the DirectML DirectX 12 backend for AMD and Intel GPUs.

In [ ]:
# Install core packages for AMD GPU PyTorch acceleration and Diffusers
!pip install --upgrade pip
!pip install torch-directml
!pip install diffusers transformers accelerate safetensors

# Install document pipeline utilities
!pip install opencv-python Pillow numpy PyQt5 PyMuPDF paddleocr

## Step 2: Initialize Stable Diffusion on AMD GPU (DirectML)
Here we load the standard `StableDiffusionImg2ImgPipeline` using the `runwayml/stable-diffusion-v1-5` base model and bind it to the DirectML device.

In [ ]:
import torch
import torch_directml
from diffusers import StableDiffusionImg2ImgPipeline

# 1. Initialize DirectML device targeting the AMD GPU
dml_device = torch_directml.device()
print(f"Using AMD GPU acceleration via DirectML device: {dml_device}")

# 2. Load the Stable Diffusion v1.5 pipeline in float32 (safest precision for DirectML)
model_id = "runwayml/stable-diffusion-v1-5"
print(f"Loading model '{model_id}' onto AMD GPU...")
sd_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32
)
sd_pipeline = sd_pipeline.to(dml_device)
print("Stable Diffusion pipeline loaded successfully!")

## Step 3: Local Image-to-Image Generation Function
This function crops the target block, runs it through the Stable Diffusion model, and applies a `strength` parameter. 

*   **`strength` (e.g. 0.35 - 0.45)**: Ensures the model retains the original texture, lighting, background noise, and structural shape of the document block while altering details to match your prompt.

In [ ]:
from PIL import Image

def generate_similar_image_local(block_image, prompt, strength=0.45, guidance_scale=7.5, steps=30):
    """
    Generates a replacement image matching the style, background noise, and dimensions of the original block.
    """
    init_image = block_image.convert("RGB")
    orig_w, orig_h = init_image.size
    
    # Scale block dimensions to multiples of 8 for Stable Diffusion
    target_w = max(256, min(768, ((orig_w + 7) // 8) * 8))
    target_h = max(256, min(768, ((orig_h + 7) // 8) * 8))
    
    resized_init = init_image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    
    print(f"Local Gen: '{prompt}' | Dimensions: {target_w}x{target_h} | Strength: {strength}")
    with torch.no_grad():
        output = sd_pipeline(
            prompt=prompt,
            image=resized_init,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=steps
        )
        
    generated_img = output.images[0]
    # Resize back to exact block bounds
    return generated_img.resize((orig_w, orig_h), Image.Resampling.LANCZOS)

## Step 4: The Document Processing Pipeline & Helpers
We now write the full implementation of the layout parser, text segmenter, region eraser, and PyQt5 renderer. 

We also add `draw_block_overlay` to draw a layout map showing where each block is located and what its `block_id` is. Any image replacements prefixed with `prompt:` will automatically call our local AMD GPU-accelerated Stable Diffusion pipeline, cropping the original block's coordinates to preserve the document's aesthetic.

In [ ]:
import os
import sys
import cv2
import json
import copy
import fitz
import numpy as np
import urllib.request
from pathlib import Path

# Define global variables
DEFAULT_FONTS_DIR = Path("./fonts")
DEFAULT_FONTS_DIR.mkdir(exist_ok=True)

def merge_bboxes(bboxes):
    if not bboxes: return None
    bboxes = np.array(bboxes)
    return [
        int(np.min(bboxes[:, 0])),
        int(np.min(bboxes[:, 1])),
        int(np.max(bboxes[:, 2])),
        int(np.max(bboxes[:, 3]))
    ]

def detect_dominant_language(data):
    if not data or "parsing_res_list" not in data: return "en"
    script_counts = {}
    for block in data.get("parsing_res_list", []):
        content = block.get("block_content")
        if not content: continue
        for ch in str(content):
            cp = ord(ch)
            if 0x0900 <= cp <= 0x097F: script_counts["hi"] = script_counts.get("hi", 0) + 1
            elif 0x0980 <= cp <= 0x09FF: script_counts["ben"] = script_counts.get("ben", 0) + 1
            elif 0x0A80 <= cp <= 0x0AFF: script_counts["gu"] = script_counts.get("gu", 0) + 1
            elif 0x0B80 <= cp <= 0x0BFF: script_counts["ta"] = script_counts.get("ta", 0) + 1
            elif 0x0C00 <= cp <= 0x0C7F: script_counts["te"] = script_counts.get("te", 0) + 1
            elif 0x0C80 <= cp <= 0x0CFF: script_counts["kn"] = script_counts.get("kn", 0) + 1
            elif 0x0D00 <= cp <= 0x0D7F: script_counts["mal"] = script_counts.get("mal", 0) + 1
            elif 0x0600 <= cp <= 0x06FF or 0x0750 <= cp <= 0x077F or 0x08A0 <= cp <= 0x08FF:
                script_counts["ar"] = script_counts.get("ar", 0) + 1
    if not script_counts: return "en"
    return max(script_counts, key=script_counts.get)

def create_text_ocr(lang="en"):
    from paddleocr import PaddleOCR
    option_sets = [
        {"use_doc_orientation_classify": False, "use_doc_unwarping": False, "use_textline_orientation": False, "lang": lang},
        {"use_angle_cls": False, "lang": lang},
        {"lang": lang}
    ]
    for options in option_sets:
        try:
            return PaddleOCR(**options)
        except Exception:
            continue
    if lang != "en":
        return create_text_ocr(lang="en")
    return PaddleOCR(use_angle_cls=False, lang="en")

def merge_same_line_entries(entries):
    if not entries: return []
    sorted_entries = sorted(entries, key=lambda e: e["bbox"][1])
    lines = []
    for entry in sorted_entries:
        bbox = entry["bbox"]
        h = bbox[3] - bbox[1]
        if h <= 0: continue
        matched_line_idx = None
        for idx, line in enumerate(lines):
            ly1, ly2 = line["y1"], line["y2"]
            y1, y2 = bbox[1], bbox[3]
            overlap = max(0, min(ly2, y2) - max(ly1, y1))
            min_h = min(ly2 - ly1, y2 - y1)
            if min_h > 0 and (overlap / min_h) >= 0.5:
                matched_line_idx = idx
                break
        if matched_line_idx is not None:
            lines[matched_line_idx]["entries"].append(entry)
            lines[matched_line_idx]["y1"] = min(lines[matched_line_idx]["y1"], bbox[1])
            lines[matched_line_idx]["y2"] = max(lines[matched_line_idx]["y2"], bbox[3])
        else:
            lines.append({"y1": bbox[1], "y2": bbox[3], "entries": [entry]})
    merged_entries = []
    for line in lines:
        line_entries = sorted(line["entries"], key=lambda e: e["bbox"][0])
        texts = [e["text"] for e in line_entries]
        merged_text = " ".join(texts)
        bboxes = [e["bbox"] for e in line_entries]
        merged_bbox = merge_bboxes(bboxes)
        scores = [e["score"] for e in line_entries if e.get("score") is not None]
        avg_score = sum(scores) / len(scores) if scores else 1.0
        merged_entries.append({"text": merged_text, "bbox": merged_bbox, "score": avg_score})
    return sorted(merged_entries, key=lambda e: (e["bbox"][1], e["bbox"][0]))

def split_multiline_blocks_with_text_ocr(data, source_image, output_dir, run_label):
    multiline_blocks = [
        block for block in data["parsing_res_list"]
        if block.get("block_content") and str(block.get("block_content")).strip()
        and str(block.get("block_label", "")).lower() not in (
            "table", "image", "header_image", "footer_image", "figure", "table_text", "seal", "chart"
        )
    ]
    if not multiline_blocks: return data
    split_data = copy.deepcopy(data)
    scale_x = source_image.width / data["width"]
    scale_y = source_image.height / data["height"]
    back_scale_x = data["width"] / source_image.width
    back_scale_y = data["height"] / source_image.height

    lang = detect_dominant_language(data)
    text_ocr = create_text_ocr(lang=lang)
    
    max_block_id = max([int(block.get("block_id", -1)) for block in split_data["parsing_res_list"]] or [-1])
    next_block_id = max_block_id + 1
    new_blocks = []
    multiline_ids = {block.get("block_id") for block in multiline_blocks}

    for block in split_data["parsing_res_list"]:
        block_id = block.get("block_id")
        if block_id not in multiline_ids:
            new_blocks.append(block)
            continue
        content = block.get("block_content")
        bbox = block.get("block_bbox")
        if not content or not bbox or len(bbox) != 4:
            new_blocks.append(block)
            continue
            
        x1 = max(0, int(bbox[0] * scale_x) - 24)
        y1 = max(0, int(bbox[1] * scale_y) - 6)
        x2 = min(source_image.width, int(bbox[2] * scale_x) + 24)
        y2 = min(source_image.height, int(bbox[3] * scale_y) + 6)
        crop = source_image.crop((x1, y1, x2, y2))
        
        try:
            temp_crop_path = output_dir / f"temp_crop_{block_id}.png"
            crop.save(temp_crop_path)
            raw_result = text_ocr.ocr(str(temp_crop_path), cls=False)
            temp_crop_path.unlink(missing_ok=True)
            
            from app_pipeline import extract_text_ocr_entries
            entries = extract_text_ocr_entries(raw_result)
            entries = merge_same_line_entries(entries)
        except Exception:
            entries = []

        if len(entries) > 1:
            for entry in entries:
                cb = entry["bbox"]
                mapped_bbox = [
                    int((x1 + cb[0]) * back_scale_x),
                    int((y1 + cb[1]) * back_scale_y),
                    int((x1 + cb[2]) * back_scale_x),
                    int((y1 + cb[3]) * back_scale_y),
                ]
                new_blocks.append({
                    "block_label": block.get("block_label", "text"),
                    "block_content": entry["text"],
                    "block_bbox": mapped_bbox,
                    "block_id": next_block_id,
                    "group_id": next_block_id,
                    "global_block_id": next_block_id,
                    "global_group_id": next_block_id,
                    "source_multiline_block_id": block_id,
                    "ocr_score": entry.get("score") or 1.0
                })
                next_block_id += 1
        else:
            new_blocks.append(block)
            
    split_data["parsing_res_list"] = new_blocks
    return split_data

def draw_block_overlay(data, source_image, output_path):
    """
    Draws a visual guide overlay with block bounding boxes and block IDs,
    saving the result as a PNG image to display inline in Jupyter cells.
    """
    img_cv = cv2.cvtColor(np.array(source_image).copy(), cv2.COLOR_RGB2BGR)
    img_h, img_w = img_cv.shape[:2]
    scale_x = img_w / data["width"]
    scale_y = img_h / data["height"]
    
    for block in data["parsing_res_list"]:
        if "block_id" not in block: continue
        bid = block["block_id"]
        bbox = block["block_bbox"]
        
        x1, y1 = int(bbox[0] * scale_x), int(bbox[1] * scale_y)
        x2, y2 = int(bbox[2] * scale_x), int(bbox[3] * scale_y)
        
        # Draw cyan bounding box
        cv2.rectangle(img_cv, (x1, y1), (x2, y2), (255, 0, 0), 2, lineType=cv2.LINE_AA)
        
        # Draw solid background for label readability
        label = f"ID {bid}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(img_cv, (x1, y1 - h - 6), (x1 + w + 4, y1), (255, 0, 0), -1)
        cv2.putText(
            img_cv, label, (x1 + 2, y1 - 3), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, lineType=cv2.LINE_AA
        )
        
    cv2.imwrite(str(output_path), img_cv)

def draw_replacements_qt(image_path, ocr_data, replacements, output_png, fonts_dir):
    os.environ["QT_QPA_FONTDIR"] = str(fonts_dir)
    os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
    from PyQt5.QtCore import Qt, QRect
    from PyQt5.QtGui import QColor, QFont, QFontDatabase, QFontMetrics, QImage, QPainter
    from PyQt5.QtWidgets import QApplication
    
    app = QApplication.instance() or QApplication(sys.argv)
    image = QImage(str(image_path))
    scale_x = image.width() / ocr_data["width"]
    scale_y = image.height() / ocr_data["height"]
    
    painter = QPainter()
    painter.begin(image)
    try:
        painter.setRenderHint(QPainter.TextAntialiasing)
        painter.setRenderHint(QPainter.Antialiasing)
        painter.setPen(QColor(0, 0, 0))
        
        for block in ocr_data["parsing_res_list"]:
            block_id = block.get("block_id")
            if block_id is None or block_id not in replacements: continue
            
            bbox = block["block_bbox"]
            x1, y1 = int(bbox[0] * scale_x), int(bbox[1] * scale_y)
            x2, y2 = int(bbox[2] * scale_x), int(bbox[3] * scale_y)
            box_w, box_h = x2 - x1, y2 - y1
            
            replacement_val = replacements[block_id]
            is_image = False
            img_filename = None
            
            # INTERCEPT PROMPT FOR LOCAL GENERATION ON AMD GPU
            if isinstance(replacement_val, str) and replacement_val.startswith("prompt:"):
                pil_source = Image.open(image_path)
                orig_crop = pil_source.crop((x1, y1, x2, y2))
                gen_prompt = replacement_val[len("prompt:"):]
                
                print(f"Generating block {block_id} locally on AMD GPU...")
                generated_img = generate_similar_image_local(orig_crop, gen_prompt, strength=0.45)
                
                img_filename = f"local_gen_{block_id}.png"
                generated_img.save(img_filename)
                is_image = True
            elif isinstance(replacement_val, str) and replacement_val.startswith("image:"):
                img_filename = replacement_val[len("image:"):]
                is_image = True
                
            if is_image and img_filename:
                img_to_draw = QImage(img_filename)
                painter.drawImage(QRect(x1, y1, box_w, box_h), img_to_draw)
                if Path(img_filename).exists() and "local_gen_" in img_filename:
                    Path(img_filename).unlink(missing_ok=True) # clean temp file
                continue
                
            # Otherwise, render as text
            from app_pipeline import get_font_family, normalize_replacement_lines
            text = str(replacement_val)
            font_family = get_font_family(text, {"default": app.font().family()})
            font_size = max(int(box_h * 0.48), 8)
            font = QFont(font_family, font_size)
            painter.setFont(font)
            painter.drawText(x1, y1, box_w, box_h, Qt.TextWordWrap, text)
    finally:
        painter.end()
    image.save(str(output_png))

## Step 5: Load Document and Generate Bounding Box Overlay map
We load the uploaded PAN card image, run the text segmentation optimizer, and generate a layout map image with annotated **Block IDs** to show you exactly which regions can be targeted for text or image overrides.

In [ ]:
from IPython.display import display, Image as DisplayImage
from app_pipeline import load_ocr_data

# 1. Define paths to your uploaded document image and OCR cache
input_image_path = Path("./document_store/pan_S/source_rendered_0.png")
raw_ocr_json = Path("./document_store/pan_S/source_rendered_0_res.json")
output_dir = Path("./output_test")
output_dir.mkdir(exist_ok=True)

if not input_image_path.exists():
    raise FileNotFoundError(f"Document image not found at {input_image_path}")

# 2. Load layout structures and split multiline rows
ocr_data = load_ocr_data(raw_ocr_json)
source_img = Image.open(input_image_path)
split_data = split_multiline_blocks_with_text_ocr(ocr_data, source_img, output_dir, "split")

# 3. Draw layout map showing all detected Block IDs
overlay_png = output_dir / "block_layout_map.png"
draw_block_overlay(split_data, source_img, overlay_png)

# 4. Display inline in the notebook cell
print("===========================================================")
print("LAYOUT ANALYSIS COMPLETE! Available blocks listed below:")
print("===========================================================")
for block in split_data["parsing_res_list"]:
    bid = block.get("block_id")
    content = block.get("block_content")
    label = block.get("block_label")
    if content and str(content).strip():
        print(f"ID {bid:2d} | Label: {label:10s} | Content: {repr(content)}")
    elif label in ("image", "header_image", "figure"):
        print(f"ID {bid:2d} | Label: {label:10s} | [Image/Graphic Area]")

print("\nSee annotated layout map below to match IDs with coordinates:")
display(DisplayImage(str(overlay_png)))

## Step 6: Enter Replacements and Render Outputs Inline
Use the block listing and layout map above to write your replacements in the `replacements` dictionary. 

*   To replace text, enter the string value directly (e.g. `21: "FATHER'S NAME"`).
*   To replace/generate graphics using the local AMD GPU-accelerated Stable Diffusion, use the **`prompt:`** prefix (e.g. `10: "prompt: a portrait..."`).

Run the cell below to perform edits and view the completed image directly in Jupyter.

In [ ]:
# Write your replacements here based on the IDs shown in Step 5
replacements = {
    20: "SHAIK KHAJA SAIF AZAM",
    21: "FATHER'S NAME",
    22: "GHOUSE BASHA SHAIK",
    
    # Replace/regenerate the profile picture block (ID 10) locally on AMD GPU:
    10: "prompt: a realistic passport photo of a smiling professional woman, high detail, studio portrait background, noise"
}

# Output file path
final_output_png = output_dir / "final_document_edited.png"

# Perform rendering (and local SD image generation)
print("Starting rendering process...")
draw_replacements_qt(input_image_path, split_data, replacements, final_output_png, DEFAULT_FONTS_DIR)
print(f"Edits completed successfully! Document saved to {final_output_png}")

# Display the final rendered document inline
display(DisplayImage(str(final_output_png)))